In [1]:
# 6 — Score a real run (.npz)

import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../.."))

from analysis.scoring import (
    scoring_input_from_npz,
    score_2d,
    score_run,
    hex_gridness_2d,
    global_order_1cell,
    autocorrelation_1cell,
)
from analysis.prototypes import reference_scores

Load and inspect the file

In [2]:
PROJECT_ROOT = Path(os.path.abspath("../.."))  # same anchor as sys.path insert
RUN = PROJECT_ROOT / "notebooks/path_integration/runs/arena2d_T40000_kap10.0_seed0.npz" 

d  = np.load(RUN)
wp = d["world_pos"]
S  = d["S_tot_buffer"]
v = d["v_body_seq"]   
torus_gt   = d["torus_gt"]       
theta_hist = d["theta_hist"]    
T  = len(wp)
step = np.median(np.linalg.norm(np.diff(wp[:, :2], axis=0), axis=1))

print(f"keys        : {list(d.files)}")
print(f"world_pos   : {wp.shape}   S_tot_buffer: {S.shape}  (T={T}, {S.shape[1]} cells)")
print(f"x range     : [{wp[:,0].min():.3f}, {wp[:,0].max():.3f}]")
print(f"y range     : [{wp[:,1].min():.3f}, {wp[:,1].max():.3f}]")
print(f"z std (flat run or not): {wp[:,2].std():.3e}")
print(f"median step : {step:.4f}")

keys        : ['world_pos', 'torus_gt', 'v_body_seq', 'theta_hist', 'n_hat_hist', 'gap_hist', 'norm_error', 'record_stride', 'S_tot_buffer']
world_pos   : (4000, 3)   S_tot_buffer: (4000, 250047)  (T=4000, 250047 cells)
x range     : [-0.999, 0.999]
y range     : [-1.000, 0.949]
z std (flat run or not): 0.000e+00
median step : 0.0076


In [3]:
si = scoring_input_from_npz(RUN, n_neuron=5000, norm="divmax")
print(f"active cells : {si.meta['n_active']}/{si.meta['n_total']} "
      f"(dead/flat traces dropped before scoring)")

IndexError: index 5044 is out of bounds for axis 1 with size 5000

Check if we have grid in the spesific run

In [ ]:
out = score_2d(RUN, n_neuron=5000, norm="minmax")
hgs, sgs = out["hgs"], out["sgs"]
grid_like = np.isfinite(hgs) & (hgs > 0.3) & (sgs < hgs)
n_act = out["f"].shape[-1]

print(f"occupancy: {out['occupancy']:.0%}   (low coverage -> weak rings; T={T} is short)")
print(f"ring detected: {out['ring_frac']:.0%} of active cells")
print(f"grid-like cells: {grid_like.sum()}/{n_act}  (HGS > 0.3 and SGS < HGS)")
if np.isfinite(hgs).any():
    print(f"median HGS (all) : {np.nanmedian(hgs):.2f}   "
              f"median HGS (grid-like): {np.nanmedian(hgs[grid_like]) if grid_like.any() else float('nan'):.2f}")
verdict = ("PASS — the CAN produces hexagonal grids; a later 3D floor would be meaningful"
               if grid_like.mean() > 0.3 else
               "WEAK a few grid-like cells; check the network/tuning (or the short trajectory) "
               "before trusting any 3D result")
print(f"\nBASELINE VERDICT : {verdict}")

In [ ]:
# show the most grid-like cells: rate map + autocorrelogram
order = np.argsort(np.where(np.isfinite(hgs), hgs, -np.inf))[::-1][:3]
fig, axes = plt.subplots(len(order), 2, figsize=(6, 2.6 * len(order)))
axes = np.atleast_2d(axes)
for row, k in enumerate(order):
    axes[row, 0].imshow(out["f"][..., k].T, origin="lower", cmap="viridis")
    axes[row, 0].set_ylabel(f"cell {k}\nHGS={hgs[k]:.2f}", fontsize=9)
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
    axes[row, 1].imshow(out["ac"][..., k].T, origin="lower", cmap="viridis")
    axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])
    if row == 0:
        axes[row, 0].set_title("rate map", fontsize=10)
        axes[row, 1].set_title("autocorr (6-fold ring = grid)", fontsize=10)

plt.suptitle("Most grid-like cells in the run", y=1.001)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
from metrics import wrapped_angle_diff

W = 200
gt_d  = wrapped_angle_diff(torus_gt[W:],   torus_gt[:-W])[:, :2]
dec_d = wrapped_angle_diff(theta_hist[W:], theta_hist[:-W])[:, :2]
sg, sd = np.linalg.norm(gt_d,1), np.linalg.norm(dec_d,1)
print("windowed cos:", np.median((dec_d*gt_d).sum(1)/(sd*sg+1e-9)))
m  = exp.qan.manifold.metric
sc = cfg.experiment.scale

# (a) the two scale candidates + what to_phase actually is (linear? a twist matrix?)
print("scale used       :", sc)
print("2pi/grid_spacing :", 2*np.pi/cfg.experiment.grid_spacing, " <- correct")
print("2pi/env_size     :", 2*np.pi/cfg.experiment.env_size,     " <- 'bug' value")
for vv in (np.array([[1.,0.]]), np.array([[0.,1.]]), np.array([[2.,0.]])):
    print("  to_phase", vv.ravel(), "->", np.round(m.to_phase(vv).ravel(), 4))

# (b) did the bump integrate at the right SPEED and DIRECTION? -> tests BOTH pasted bugs
gt_vel  = wrapped_angle_diff(torus_gt[1:],   torus_gt[:-1])[:, :2]   # how it SHOULD move
dec_vel = wrapped_angle_diff(theta_hist[1:], theta_hist[:-1])[:, :2] # how it ACTUALLY moved
sp_gt, sp_dec = np.linalg.norm(gt_vel,1), np.linalg.norm(dec_vel,1)
mask = sp_gt > 1e-4
print("speed ratio decoded/gt :", round(np.median(sp_dec[mask]/sp_gt[mask]),3),
      " (1=correct, ~.15=6.7x bug, ~.88=gain65)")
cos = (dec_vel*gt_vel).sum(1)/(sp_dec*sp_gt+1e-12)
print("direction cos(dec,gt)  :", round(np.median(cos[mask]),3),
      " (1=aligned, <1=mis-rotated => twist)")

# (c) what KIND of tracking error: constant offset vs accumulating drift?
e = np.linalg.norm(wrapped_angle_diff(theta_hist, torus_gt), axis=1)
print("err t<500:", round(e[:500].mean(),3), "| t~20k:", round(e[19750:20250].mean(),3),
      "| t>39.5k:", round(e[-500:].mean(),3), "| mean as %period:", round(e.mean()/(2*np.pi),3))

# (d) coverage, restated as the cause of the gridness collapse
print("x span:", round(wp[:,0].ptp(),3), " y span:", round(wp[:,1].ptp(),3),
      " box:", cfg.experiment.env_size)